In [1]:
# Train YOLOv8 Model for Tomato Leaf Disease Detection
!pip install torch torchvision torchaudio --index-url https://download.pytorch.org/whl/cu124 -q
!pip install ultralytics matplotlib -q
print("✅ Dependencies installed!")


✅ Dependencies installed!


In [2]:
import torch
from ultralytics import YOLO
import os
import matplotlib.pyplot as plt

# Check GPU availability
device = 0 if torch.cuda.is_available() else 'cpu'
if device == 0:
    print(f"✅ GPU Available: {torch.cuda.get_device_name(0)}")
else:
    print("⚠️ GPU not available, using CPU (training will be slow)")
print(f"Device to use: {device}\n")


✅ GPU Available: NVIDIA GeForce RTX 2050
Device to use: 0



In [3]:
# Verify dataset structure
dataset_path = "./Tomato-Leaf-Disease-63"
print(f"Checking dataset at: {dataset_path}")
print(f"Dataset exists: {os.path.exists(dataset_path)}")

train_images = f"{dataset_path}/train/images"
val_images = f"{dataset_path}/valid/images"
test_images = f"{dataset_path}/test/images"

print(f"\nTrain images: {len(os.listdir(train_images)) if os.path.exists(train_images) else 0} files")
print(f"Validation images: {len(os.listdir(val_images)) if os.path.exists(val_images) else 0} files")
print(f"Test images: {len(os.listdir(test_images)) if os.path.exists(test_images) else 0} files")


Checking dataset at: ./Tomato-Leaf-Disease-63
Dataset exists: True

Train images: 9037 files
Validation images: 843 files
Test images: 164 files


In [4]:
# Create/Verify data.yaml configuration
data_yaml = f"""
path: {os.path.abspath(dataset_path)}
train: train/images
val: valid/images
test: test/images

nc: 9
names: [
  'Early_Blight', 'Healthy', 'Late_Blight', 'Leaf_Miner',
  'Leaf_Mold', 'Mosaic_Virus', 'Septoria_Leaf_Spot',
  'Spider_Mites', 'Yellow_Leaf_Curl_Virus'
]
"""

with open("data.yaml", "w") as f:
    f.write(data_yaml)
print("✅ data.yaml configuration created")


✅ data.yaml configuration created


In [5]:
# Load YOLOv8 model and start training
print("Loading YOLOv8n pre-trained model...")
model = YOLO('yolov8n.pt')
print("✅ Model loaded")

print("\n" + "="*60)
print("Starting training... This may take some time!")
print("="*60 + "\n")

# Train the model
results = model.train(
    data='data.yaml',
    epochs=50,
    imgsz=640,
    batch=16,  # Adjust based on your GPU memory
    device=device,
    name='tomato_disease_notebook',
    patience=10,
    save=True,
    verbose=True
)

print("\n✅ Training complete!")


Loading YOLOv8n pre-trained model...
✅ Model loaded

Starting training... This may take some time!

New https://pypi.org/project/ultralytics/8.4.19 available  Update with 'pip install -U ultralytics'
Ultralytics 8.4.18  Python-3.11.7 torch-2.6.0+cu124 CUDA:0 (NVIDIA GeForce RTX 2050, 4096MiB)
engine\trainer: agnostic_nms=False, amp=True, angle=1.0, augment=False, auto_augment=randaugment, batch=16, bgr=0.0, box=7.5, cache=False, cfg=None, classes=None, close_mosaic=10, cls=0.5, compile=False, conf=None, copy_paste=0.0, copy_paste_mode=flip, cos_lr=False, cutmix=0.0, data=data.yaml, degrees=0.0, deterministic=True, device=0, dfl=1.5, dnn=False, dropout=0.0, dynamic=False, embed=None, end2end=None, epochs=50, erasing=0.4, exist_ok=False, fliplr=0.5, flipud=0.0, format=torchscript, fraction=1.0, freeze=None, half=False, hsv_h=0.015, hsv_s=0.7, hsv_v=0.4, imgsz=640, int8=False, iou=0.7, keras=False, kobj=1.0, line_width=None, lr0=0.01, lrf=0.01, mask_ratio=4, max_det=300, mixup=0.0, mode=t

KeyboardInterrupt: 

In [ ]:
# Display results metrics — auto-detect latest run folder
from pathlib import Path
run_dir = Path('runs/detect')
matching = sorted(run_dir.glob('tomato_disease_notebook*'), key=os.path.getmtime)
latest_run = str(matching[-1]) if matching else 'runs/detect/tomato_disease_notebook'

print("\n" + "="*60)
print("TRAINING SUMMARY")
print("="*60)
print(f"Results saved to: {latest_run}/")
print(f"\nMetrics available:")
print(f"  - Weights: {latest_run}/weights/best.pt")
print(f"  - Results: {latest_run}/results.png")


In [ ]:
# Load and display results image
results_path = f'{latest_run}/results.png'
if os.path.exists(results_path):
    print(f"\n✅ Loading results metrics image...")
    plt.figure(figsize=(15, 10))
    img = plt.imread(results_path)
    plt.imshow(img)
    plt.axis('off')
    plt.title('Training Metrics')
    plt.tight_layout()
    plt.show()
else:
    print(f"⚠️ Results image not found at {results_path}")


In [ ]:
# Test the trained model on a test image
print("\n" + "="*60)
print("TESTING MODEL ON TEST IMAGE")
print("="*60)

best_weights = f'{latest_run}/weights/best.pt'
if os.path.exists(best_weights):
    print(f"✅ Loading trained model from: {best_weights}")
    trained_model = YOLO(best_weights)

    test_img_path = './Tomato-Leaf-Disease-63/test/images'
    if os.path.exists(test_img_path):
        sample_images = [os.path.join(test_img_path, f) for f in os.listdir(test_img_path) if f.endswith(('.jpg', '.png'))]

        if sample_images:
            print(f"Found {len(sample_images)} test images")

            # Test on first 3 images
            for idx, test_image in enumerate(sample_images[:3]):
                print(f"\n🔍 Testing image {idx+1}/{min(3, len(sample_images))}: {os.path.basename(test_image)}")
                results = trained_model.predict(source=test_image, conf=0.5)

                for r in results:
                    print(f"   - Detections: {len(r.boxes)} objects found")
                    im_array = r.plot()

                    plt.figure(figsize=(10, 8))
                    plt.imshow(im_array[..., ::-1])  # Convert BGR to RGB
                    plt.axis('off')
                    plt.title(f'Detection Results - {os.path.basename(test_image)}')
                    plt.tight_layout()
                    plt.show()
        else:
            print("⚠️ No test images found")
    else:
        print(f"⚠️ Test image path does not exist: {test_img_path}")
else:
    print(f"⚠️ Trained weights not found at: {best_weights}")

print("\n✅ All done! Your model is trained and ready for inference.")
